## Constructor

### Construct can overload

```cpp
class StanfordID {
private:
    std::string name;
    int idNumber;

public:
    // "Default Constructor" (takes no arguments)
    StanfordID() {
        name = "Unknown";
        idNumber = 0;
    }

    // "Parameterized Constructor"
    StanfordID(std::string n, int id) {
        name = n;
        idNumber = id;
    }
};
```

### Uniform Initialization

Necessity: Some types of data, like const (constant) variables, must be initialized this way because they aren't allowed to be changed once the body of the constructor starts.

```cpp
StanfordID(string n, int id) : name(n), idNumber(id) {
    // The body is now empty!
}
```

### in header file

Must include the class scope resolution. Just like `std::` (which is a class 

having header file: `bits/stdc++.h`), but you can't use `using namespace std`

```cpp
// .cpp file (implementation)
#include "StanfordID.h"
using namespace std; // but can't using namespace StanfordID

StanfordID::StanfordID(string name, string sunet, int idNumber) :
    name{name}, sunet{sunet}, idNumber{idNumber} 
{
  // The body is now empty unless you need to run specific logic!
}
```

## Destructor

- **Syntax**: It has the exact same name as the class, but it is preceded by a tilde `~` (e.g., `~StanfordID()`).

- **No Parameters**: Unlike constructors, destructors cannot take any arguments

- **Automatic Execution**: C++ calls it automatically as soon as an object "goes out of scope" (like when a function ends or a program finishes)

- **When You MUST Write One**: if your constructor uses the `new` keyword to grab memory, your destructor **must** use the `delete` keyword to give it back.


```cpp
~MyClass() {
    delete my_ptr; // Deallocates a single object
}

~StanfordID() {
    delete [] my_array; // Correct syntax for an array 
}

```

- **Smart Pointers**: modern C++ often uses "Smart Pointers" (`std::shared_ptr<T>` or `std::unique_ptr<T>`) to handle this cleanup automatically.

## Beware of input arg same as class field


If your function is void f(std::string name), the compiler follows a strict "hierarchy of search":

- Local scope first: It looks for a variable named `name` inside the function's arguments or the body of f.

- Class scope second: If it doesn't find a local one, it looks at the class fields.

```cpp
void StanfordID::setName(std::string name) {
    // name = name;       // This does nothing!
    this->name = name;    // This works
}
```

`this->` to solve the conflict applies universally.

- The `this->` syntax is **not** about the type of the variable you are accessing (like an `int` or `std::string`); it is about the fact that `this` itself is a **pointer to the object**.

- Because `this` is a pointer to the current instance of the class, you use `this->` to tell the compiler: "Look inside the object I am currently pointing to and find this field".

- No `.h` File or Imports Required
You do not need to import anything or have a `.h` file to use `this`.

## Inherence

```cpp
// Base Class
class Shape {
    // ...
};

// Subclass inheriting from Shape
class Circle : public Shape { 
    // Circle now has access to Shape's public features!
};
```

### Virtual Function

1. When we want every shape to have an area, but the generic Shape class doesn't know how to calculate it, we use a Pure Virtual Function.

2. This is a special keyword used only for functions. It tells C++ that if a child class (like Circle) has its own version of a function (like area()), C++ should use the child's version even if we are looking at it through a parent's eyes

3. Why not `public` or `private` ? 
    - If we made it public but not virtual, then the computer would always try to run the "Base" version of the code instead of the specific "Circle" version.
    
    - If we made area() private, then a Circle wouldn't even be allowed to know that the Shape class wants it to have an area!

- The Abstract Base Class (Shape)

    In your header file, you define the "blueprint." Adding `virtual` and `= 0` tells C++, "This function exists, but I'm not writing the logic here—the children must do it"

    ```cpp
    // Shape.h
    class Shape {
    public:
        // Pure virtual function
        virtual double area() const = 0; 
    };    
    ```

- The Implementation (Circle)

    When we create Circle, we "override" that area function with specific math.
    ```cpp
    class Circle : public Shape {
    public:
        // Constructor using the list initialization you like!
        Circle(double radius) : _radius{radius} {} 

        // overwrite the base class function area() for a circle
        double area() const {
            return 3.14 * _radius * _radius;
        }

    private:
        double _radius;
    };
    ```

### 3 Types of Inheritance

That `public` keyword before the parent's name defines the Type of Inheritance. This determines what the "outside world" can see of the parent's features once they are handed down to the child .

- public: accessible everywhere the object is visible

- protected: accessible inside the class and its derived classes
not accessible from normal outside code

| Inheritance | Public in A becomes | Protected in A becomes |
|---|---|---|
| `public` | `public` | `protected` |
| `protected` | `protected` | `protected` |
| `private` | `private` | `private` |

### Diamond problem

1. When A being parent of B and C, and D inherits from both B and C (aka A being grandparent of D)

2. because B and C both inherit from A, they each call the constructor of A. 

3. When D inherits from both of them, it ends up with two separate copies of class A inside of it—one from B and one from C

    ```cpp
    D obj;
    obj.hello(); // WHOSE method do I call???
    ```

Solution: Virtual Inheritance

1. To solve this, we use Virtual Inheritance: add the virtual keyword to the inheritance line for the middle classes (B and C). 

2. This tells C++ that a derived class should only have a single instance of the base class, as it creates a shared instance of A between B and C.

```cpp
class A {
public:
    void hello() { /* ... */ }
};

// Even in one file, you MUST use 'virtual' here
class B : virtual public A { }; 
class C : virtual public A { };

class D : public B, public C { };

int main() {
    D obj;
    obj.hello(); // Fixed! No longer ambiguous
}
```